# ETL de limpieza de datos para subida de archivos a base de datos


### Importacion de librerias

In [101]:
import pandas as pd

In [102]:
ruta_destino = 'C:/Users/nixon/Desktop/data-projects/automotive_dw_project/data/processed/car_sales_cleaned.csv'

### lectura del dataset

In [103]:
df = pd.read_csv('../data/raw/car_sales.csv')
df.head()


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


### Revision de nulos


In [104]:
df.isnull().sum()

year                0
make            10301
model           10399
trim            10651
body            13195
transmission    65352
vin                 4
state               0
condition       11820
odometer           94
color             749
interior          749
seller              0
mmr                38
sellingprice       12
saledate           12
dtype: int64

In [105]:
df = df.dropna(subset=['model', 'year', 'state', 'vin', 'sellingprice','mmr','saledate'])

In [106]:
columns_to_fill = [ 'color', 'transmission', 'body', 'trim','interior']
df[columns_to_fill] = df[columns_to_fill].fillna('Unknown')

In [107]:
df['condition'] = df['condition'].astype('float')
columns_median = ['odometer', 'condition']
for column in columns_median:
    median_value = df[column].mean()
    df[column] = df[column].fillna(median_value)

- Los nulos en columnas categóricas como transmission y color fueron reemplazados por "unknown".
- Se eliminaron filas con nulos en columnas críticas como vin, selling_price y sale_date.

### Revision de duplicados

In [108]:
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

Number of duplicate rows: 0


- Se eliminaron registros duplicados para evitar inconsistencias futuras en el Data Warehouse.

### Revision de tipos de datos

In [109]:
df.dtypes

year              int64
make             object
model            object
trim             object
body             object
transmission     object
vin              object
state            object
condition       float64
odometer        float64
color            object
interior         object
seller           object
mmr             float64
sellingprice    float64
saledate         object
dtype: object

In [110]:
df['condition'] = df['condition'].astype('category')
df['saledate'] = pd.to_datetime(df['saledate'], utc=True).dt.normalize().dt.tz_localize(None)

C:\Users\nixon\AppData\Local\Temp\ipykernel_6968\2245895742.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['saledate'] = pd.to_datetime(df['saledate'], utc=True).dt.normalize().dt.tz_localize(None)


In [112]:
df.dtypes

year                     int64
make                    object
model                   object
trim                    object
body                    object
transmission            object
vin                     object
state                   object
condition             category
odometer               float64
color                   object
interior                object
seller                  object
mmr                    float64
sellingprice           float64
saledate        datetime64[ns]
dtype: object

- sale_date fue convertida a datetime para facilitar análisis temporal y futura creación de dim_time.

### Normalizacion de nombre de columnas

In [113]:
df = df.rename(columns={'saledate': 'sale_date','sellingprice': 'selling_price'})

In [114]:
df.head()

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,selling_price,sale_date
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,2014-12-16
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,2014-12-16
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,2015-01-14
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,2015-01-28
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,2014-12-18


- Los nombres de columnas fueron convertidos a snake_case para mantener consistencia en SQL y PostgreSQL.


## normalizando datos de columnas

- Guiones en columnas [color / interior] = normalizado a unknown
- Algunos nombres en body mayusculas = normalizado a minusculas
- Datos de condition incosistentes que superen 5.0 por mala digitacion

In [115]:
df['color'] = df['color'].replace('—', 'unknown')
df['interior'] = df['interior'].replace('—', 'unknown')

In [116]:
df['body'] = df['body'].str.lower()

In [117]:
df['condition'] = df['condition'].astype('float')
df.loc[df['condition'] > 5, 'condition'] = (
    df.loc[df['condition'] > 5, 'condition'] / 10
)


In [118]:
df['condition'] = df['condition'].astype('category')
df['condition'].head()

0    5.0
1    5.0
2    4.5
3    4.1
4    4.3
Name: condition, dtype: category
Categories (42, float64): [1.0, 1.1, 1.2, 1.3, ..., 4.7, 4.8, 4.9, 5.0]

In [119]:
df['year'].value_counts

<bound method IndexOpsMixin.value_counts of 0         2015
1         2015
2         2014
3         2015
4         2014
          ... 
558832    2015
558833    2012
558834    2012
558835    2015
558836    2014
Name: year, Length: 548400, dtype: int64>

## Integridad final de datos de venta

In [120]:
df.dtypes

year                      int64
make                     object
model                    object
trim                     object
body                     object
transmission             object
vin                      object
state                    object
condition              category
odometer                float64
color                    object
interior                 object
seller                   object
mmr                     float64
selling_price           float64
sale_date        datetime64[ns]
dtype: object

In [121]:
(df['selling_price'] <= 0).sum()
(df['odometer'] < 0).sum()


np.int64(0)

In [122]:
df['selling_price'].value_counts(ascending=False).head

<bound method NDFrame.head of selling_price
11000.0     4409
12000.0     4395
13000.0     4288
10000.0     3994
14000.0     3863
            ... 
37850.0        1
20925.0        1
126250.0       1
54700.0        1
27840.0        1
Name: count, Length: 1878, dtype: int64>

### Exportacion de data

In [123]:
df.to_csv(ruta_destino, index=False, encoding='utf-8')